# 01 — Stylized facts of BTCUSDT perp trades (fixture scale)

The Cont (2001) battery on the committed 1-hour fixture. At fixture scale the estimates are noisy — the full study window (Jan 2024) sharpens them; the *code path* is identical.

In [ ]:
FIXTURE_ONLY = True  # papermill parameter

In [ ]:
import numpy as np
import polars as pl
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from quantsim_research.config import repo_root

FIXTURE = repo_root() / "data" / "fixtures" / "binance-um" / "trades" / "BTCUSDT-2024-01-07-h00.csv"
ARTIFACTS = repo_root() / "artifacts" / "notebooks"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
df = pl.read_csv(FIXTURE)
ts_ns = df["ts_ns"].to_numpy()
price = df["price_e8"].to_numpy() / 1e8
side = df["side"].to_numpy()
print(f"fixture: {len(df)} trades, {(ts_ns[-1]-ts_ns[0])/1e9:.0f}s span")


## Return moments and fat tails
1-second log-returns: excess kurtosis, Jarque-Bera, Hill tail index.

In [ ]:
from quantsim_research.stylized_facts.returns import excess_kurtosis_by_scale, moments
from quantsim_research.stylized_facts.tails import hill_estimator

# Resample to a 1-second last-price grid.
sec = ((ts_ns - ts_ns[0]) / 1e9).astype(int)
last_in_sec = {}
for s, p in zip(sec, price, strict=True):
    last_in_sec[s] = p
grid = np.array([last_in_sec[k] for k in sorted(last_in_sec)])
r1 = np.diff(np.log(grid))
m = moments(r1)
jb, jb_p = m.jarque_bera()
hill = hill_estimator(r1, tail_fraction=0.05)
print(f"n={m.n}  std={m.std:.2e}  skew={m.skewness:.2f}  ex.kurt={m.excess_kurtosis:.2f}")
print(f"Jarque-Bera={jb:.0f} (p={jb_p:.1e})  Hill alpha={hill.alpha:.2f} (k={hill.k})")
agg = excess_kurtosis_by_scale(grid, [1, 5, 15, 60])
print("aggregational gaussianity (ex.kurt by scale):", {k: round(v, 2) for k, v in agg.items()})


## Volatility clustering vs no linear autocorrelation

In [ ]:
from quantsim_research.stylized_facts.autocorr import acf, bartlett_band, ljung_box

lags = 60
rho_r = acf(r1, lags)
rho_abs = acf(np.abs(r1), lags)
band = bartlett_band(r1.size)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].bar(range(1, lags + 1), rho_r)
ax[0].axhline(band, ls='--', c='r')
ax[0].axhline(-band, ls='--', c='r')
ax[0].set_title("ACF raw returns (should die fast)")
ax[1].bar(range(1, lags + 1), rho_abs); ax[1].axhline(band, ls='--', c='r')
ax[1].set_title("ACF |returns| (clustering)")
fig.tight_layout(); fig.savefig(ARTIFACTS / "01_acf.png", dpi=120)
print("Ljung-Box raw:", ljung_box(r1, 20))
print("Ljung-Box |r|:", ljung_box(np.abs(r1), 20))


## Trade-sign autocorrelation and the Zumbach asymmetry
Sign clustering is strong and slow (order splitting). Zumbach asymmetry is the statistic our linear-Hawkes simulator is *preregistered to miss* — quadratic Hawkes (Stage 2) targets it.

In [ ]:
from quantsim_research.stylized_facts.orderflow import trade_sign_autocorr, zumbach_asymmetry

sign_acf = trade_sign_autocorr(side.astype(float), max_lag=200)
print(f"sign ACF lag1={sign_acf[0]:.3f} lag10={sign_acf[9]:.3f} lag100={sign_acf[99]:.3f}")
z = zumbach_asymmetry(grid, window=30)
print(f"Zumbach: trend->vol={z.trend_predicts_vol:.3f}  vol->trend={z.vol_predicts_trend:.3f}  "
      f"asymmetry={z.asymmetry:+.3f}")
